In [147]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Set display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

## Load Staged Datasets

Let's load all 4 clean datasets from the staging layer

In [148]:
# Load staged datasets
customers = pd.read_csv("../staging/customers_staged.csv", parse_dates=["signup_date"])
usage_logs = pd.read_csv("../staging/usage_logs_staged.csv", parse_dates=["log_date"])
support_tickets = pd.read_csv(
    "../staging/support_tickets_staged.csv",
    parse_dates=["created_date", "resolved_date"],
)
churn = pd.read_csv(
    "../staging/churn_staged.csv", parse_dates=["churn_date", "observation_date"]
)

print("Loaded Staged Datasets:")
print("=" * 60)
print(f"Customers: {customers.shape}")
print(f"Usage Logs: {usage_logs.shape}")
print(f"Support Tickets: {support_tickets.shape}")
print(f"Churn: {churn.shape}")

Loaded Staged Datasets:
Customers: (5000, 7)
Usage Logs: (212936, 6)
Support Tickets: (5944, 8)
Churn: (5000, 4)


---

## Part 1: Customer Features

Extract features from the customers table

In [149]:
churn.head()

,customer_id,churn_date,churned,observation_date
0,1,NaT,0,2024-12-31
1,2,NaT,0,2024-12-31
2,3,NaT,0,2024-12-31
3,4,NaT,0,2024-12-31
4,5,NaT,0,2024-12-31


In [150]:
# Get observation date (same for all customers)
observation_date = churn["observation_date"].iloc[0]
print(f"Observation Date: {observation_date}")

# Start with customers as base
features = customers.copy()

print(f"\nStarting features dataframe: {features.shape}")
features.head()

Observation Date: 2024-12-31 00:00:00

Starting features dataframe: (5000, 7)


,customer_id,signup_date,country,age,gender,subscription_tier,monthly_fee
0,1,2023-10-17,Germany,62.00,F,Basic,13.00
1,2,2022-04-25,UK,43.50,M,Premium,30.48
2,3,2022-01-26,Brazil,53.00,Unknown,Basic,11.59
3,4,2024-01-30,Canada,26.00,M,Basic,11.97
4,5,2022-10-09,Spain,44.00,F,Basic,11.23


### Feature 1: Account age in days

How long has the customer been with us?

In [151]:
# Calculate account age
features["account_age_days"] = (observation_date - features["signup_date"]).dt.days

print(f"Account age statistics:")
print(features["account_age_days"].describe())
print(f"\nSample:")
features[["customer_id", "signup_date", "account_age_days"]].head()

Account age statistics:
count   5000.00
mean     635.96
std      260.07
min      184.00
25%      410.75
50%      635.00
75%      855.00
max     1095.00
Name: account_age_days, dtype: float64

Sample:


,customer_id,signup_date,account_age_days
0,1,2023-10-17,441
1,2,2022-04-25,981
2,3,2022-01-26,1070
3,4,2024-01-30,336
4,5,2022-10-09,814


### Feature 2: Is premium customer?

Create a binary flag for premium/enterprise tiers

In [152]:
# Create is_premium flag (Premium or Enterprise)
features["is_premium"] = (
    features["subscription_tier"].isin(["Premium", "Enterprise"]).astype(int)
)

print(f"Premium customer distribution:")
print(features["is_premium"].value_counts())
print(f"\nBy tier:")
print(pd.crosstab(features["subscription_tier"], features["is_premium"]))

Premium customer distribution:
is_premium
0    3008
1    1992
Name: count, dtype: int64

By tier:
is_premium            0     1
subscription_tier            
Basic              3008     0
Enterprise            0   516
Premium               0  1476


### Feature 3: Age group

Create age categories for segmentation

In [153]:
# Create age groups
features["age_group"] = pd.cut(
    features["age"],
    bins=[0, 25, 35, 50, 100],
    labels=["18-25", "26-35", "36-50", "51+"],
)

print(f"Age group distribution:")
print(features["age_group"].value_counts().sort_index())

Age group distribution:
age_group
18-25     708
26-35     918
36-50    1593
51+      1781
Name: count, dtype: int64


### Feature 4: Country region

Group countries into regions for better generalization

In [154]:
# Define country to region mapping
region_mapping = {
    "USA": "North America",
    "Canada": "North America",
    "Mexico": "North America",
    "UK": "Europe",
    "Germany": "Europe",
    "Spain": "Europe",
    "France": "Europe",
    "Italy": "Europe",
    "Brazil": "South America",
    "Australia": "Oceania",
}

# Create region column
features["country_region"] = features["country"].map(region_mapping)

print(f"Region distribution:")
print(features["country_region"].value_counts())

Region distribution:
country_region
Europe           2477
North America    1765
Oceania           400
South America     358
Name: count, dtype: int64


---

## Part 2: Usage Features

Aggregate usage logs to create customer-level usage patterns

### Strategy:
We'll calculate usage metrics for each customer over the observation period (last 90 days)

In [155]:
print(
    f"Usage logs date range: {usage_logs['log_date'].min()} to {usage_logs['log_date'].max()}"
)
print(f"Total usage log entries: {len(usage_logs):,}")
print(f"Unique customers with usage: {usage_logs['customer_id'].nunique():,}")

Usage logs date range: 2024-10-03 00:00:00 to 2024-12-31 00:00:00
Total usage log entries: 212,936
Unique customers with usage: 5,000


### Aggregate usage metrics per customer

We'll create multiple features from the usage logs:
- Total and average sessions
- Total and average usage time
- Feature usage patterns
- Error rates
- Activity consistency
- Recency of activity

In [ ]:
# Create usage features
usage_features = usage_logs.groupby("customer_id", as_index=False).agg(
    {
        "sessions": ["sum", "mean"],
        "duration_minutes": ["sum", "mean"],
        "features_used": "mean",
        "errors_encountered": "sum",
        "log_date": ["count", "max"],  # count = days active, max = last activity
    }
)

usage_features.head()

customer_id sessions      duration_minutes       features_used  \
                   sum mean              sum  mean          mean   
0           1      349 5.63          1745.06 28.15          8.13   
1           2      403 5.30          2515.99 33.11          7.57   
2           3      421 5.40          2312.93 29.65          7.53   
3           4      147 5.25           662.64 23.67          6.86   
4           5      263 4.61          1803.58 31.64          7.09   

  errors_encountered log_date             
                 sum    count        max  
0                 26       62 2024-12-31  
1                 39       76 2024-12-31  
2                 42       78 2024-12-31  
3                 12       28 2024-12-27  
4                 37       57 2024-12-31

In [157]:
# Flatten column names
usage_features.columns = [
    "customer_id",
    "total_sessions",
    "avg_sessions_per_day",
    "total_usage_minutes",
    "avg_session_duration",
    "avg_features_per_session",
    "total_errors",
    "days_active",
    "last_activity_date",
]

print(f"Usage features created: {usage_features.shape}")
usage_features.head()

Usage features created: (5000, 9)


,customer_id,total_sessions,avg_sessions_per_day,total_usage_minutes,avg_session_duration,avg_features_per_session,total_errors,days_active,last_activity_date
0,1,349,5.63,1745.06,28.15,8.13,26,62,2024-12-31
1,2,403,5.30,2515.99,33.11,7.57,39,76,2024-12-31
2,3,421,5.40,2312.93,29.65,7.53,42,78,2024-12-31
3,4,147,5.25,662.64,23.67,6.86,12,28,2024-12-27
4,5,263,4.61,1803.58,31.64,7.09,37,57,2024-12-31


### Calculate days since last activity

In [158]:
# Days since last activity
usage_features["days_since_last_activity"] = (
    observation_date - usage_features["last_activity_date"]
).dt.days

print(f"Days since last activity statistics:")
print(usage_features["days_since_last_activity"].describe())

Days since last activity statistics:
count   5000.00
mean       2.06
std        4.01
min        0.00
25%        0.00
50%        1.00
75%        2.00
max       47.00
Name: days_since_last_activity, dtype: float64


### Calculate usage consistency

Standard deviation of daily sessions - lower values mean more consistent usage

In [159]:
# Calculate standard deviation of sessions per customer
usage_consistency = usage_logs.groupby("customer_id")["sessions"].std().reset_index()
usage_consistency.columns = ["customer_id", "usage_consistency_std"]

# Fill NaN with 0 (customers with only 1 day of activity)
usage_consistency["usage_consistency_std"] = usage_consistency[
    "usage_consistency_std"
].fillna(0)

usage_consistency.head()

,customer_id,usage_consistency_std
0,1,2.65
1,2,2.88
2,3,2.49
3,4,2.24
4,5,2.60


In [160]:
# Merge with usage_features
usage_features = usage_features.merge(usage_consistency, on="customer_id", how="left")

# transform()

print(f"Usage consistency statistics:")
print(usage_features["usage_consistency_std"].describe())

Usage consistency statistics:
count   5000.00
mean       2.57
std        0.24
min        0.82
25%        2.45
50%        2.58
75%        2.70
max        3.79
Name: usage_consistency_std, dtype: float64


### Calculate usage trend

Is the customer's usage increasing or decreasing over time?
We'll use a simple approach: compare first 30 days vs last 30 days

In [161]:
# Calculate midpoint of observation period
midpoint_date = usage_logs["log_date"].min() + timedelta(days=45)

# Split into early and late periods
early_usage = (
    usage_logs[usage_logs["log_date"] < midpoint_date]
    .groupby("customer_id")["sessions"]
    .mean()
)
late_usage = (
    usage_logs[usage_logs["log_date"] >= midpoint_date]
    .groupby("customer_id")["sessions"]
    .mean()
)

# Calculate trend (late - early)
usage_trend = (late_usage - early_usage).reset_index()

usage_trend

,customer_id,sessions
0,1,0.83
1,2,0.67
2,3,0.59
3,4,-0.04
4,5,-0.13
...,...,...
4995,4996,-0.39
4996,4997,-0.79
4997,4998,1.67
4998,4999,-0.13


In [ ]:
usage_trend.columns = ["customer_id", "usage_trend"]
usage_trend["usage_trend"] = usage_trend["usage_trend"].fillna(0)  # No trend if missing

# Merge with usage_features
usage_features = usage_features.merge(usage_trend, on="customer_id", how="left")
usage_features["usage_trend"] = usage_features["usage_trend"].fillna(0)

print(f"Usage trend statistics (positive = increasing):")
print(usage_features["usage_trend"].describe())
print(f"\nIncreasing usage: {(usage_features['usage_trend'] > 0).sum()}")
print(f"Decreasing usage: {(usage_features['usage_trend'] < 0).sum()}")

### Drop the last_activity_date column (we have days_since_last_activity instead)

In [162]:
usage_features = usage_features.drop("last_activity_date", axis=1)

print(f"Final usage features: {usage_features.shape}")
print(f"\nColumns: {list(usage_features.columns)}")
usage_features.head()

Final usage features: (5000, 10)

Columns: ['customer_id', 'total_sessions', 'avg_sessions_per_day', 'total_usage_minutes', 'avg_session_duration', 'avg_features_per_session', 'total_errors', 'days_active', 'days_since_last_activity', 'usage_consistency_std']


,customer_id,total_sessions,avg_sessions_per_day,total_usage_minutes,avg_session_duration,avg_features_per_session,total_errors,days_active,days_since_last_activity,usage_consistency_std
0,1,349,5.63,1745.06,28.15,8.13,26,62,0,2.65
1,2,403,5.30,2515.99,33.11,7.57,39,76,0,2.88
2,3,421,5.40,2312.93,29.65,7.53,42,78,0,2.49
3,4,147,5.25,662.64,23.67,6.86,12,28,4,2.24
4,5,263,4.61,1803.58,31.64,7.09,37,57,0,2.60


---

## Part 3: Support Features

Aggregate support tickets to create customer-level support interaction metrics

In [163]:
print(f"Total support tickets: {len(support_tickets):,}")
print(f"Unique customers with tickets: {support_tickets['customer_id'].nunique():,}")
print(f"Unresolved tickets: {support_tickets['resolved_date'].isna().sum():,}")

Total support tickets: 5,944
Unique customers with tickets: 2,000
Unresolved tickets: 1,204


### Basic ticket counts and resolution metrics

In [164]:
# Basic aggregations
support_features = (
    support_tickets.groupby("customer_id")
    .agg(
        {
            "ticket_id": "count",  # Total tickets
            "resolution_time_hours": "mean",  # Average resolution time
            "created_date": "max",  # Last ticket date
        }
    )
    .reset_index()
)

support_features.columns = [
    "customer_id",
    "total_tickets",
    "avg_resolution_hours",
    "last_ticket_date",
]

print(f"Support features created: {support_features.shape}")
support_features.head()

Support features created: (2000, 4)


,customer_id,total_tickets,avg_resolution_hours,last_ticket_date
0,3,4,20.02,2024-10-22
1,4,1,12.11,2024-03-20
2,8,5,17.25,2024-10-18
3,10,1,40.44,2024-08-20
4,11,4,12.88,2024-12-25


### Count unresolved tickets per customer

In [165]:
# Count unresolved tickets
unresolved = (
    support_tickets[support_tickets["resolved_date"].isna()]
    .groupby("customer_id")
    .size()
    .reset_index()
)
unresolved.columns = ["customer_id", "unresolved_tickets"]

# Merge with support_features
support_features = support_features.merge(unresolved, on="customer_id", how="left")
support_features["unresolved_tickets"] = (
    support_features["unresolved_tickets"].fillna(0).astype(int)
)

print(
    f"Customers with unresolved tickets: {(support_features['unresolved_tickets'] > 0).sum()}"
)

Customers with unresolved tickets: 915


### Calculate percentage of technical tickets

In [166]:
# Technical tickets per customer
technical_tickets = (
    support_tickets[support_tickets["category"] == "Technical"]
    .groupby("customer_id")
    .size()
)
total_tickets = support_tickets.groupby("customer_id").size()

technical_pct = (technical_tickets / total_tickets * 100).reset_index()
technical_pct.columns = ["customer_id", "technical_tickets_pct"]

# Merge with support_features
support_features = support_features.merge(technical_pct, on="customer_id", how="left")
support_features["technical_tickets_pct"] = support_features[
    "technical_tickets_pct"
].fillna(0)

print(f"Technical tickets percentage statistics:")
print(support_features["technical_tickets_pct"].describe())

Technical tickets percentage statistics:
count   2000.00
mean      50.21
std       34.04
min        0.00
25%       25.00
50%       50.00
75%       75.00
max      100.00
Name: technical_tickets_pct, dtype: float64


### Count high priority tickets

In [167]:
# High priority tickets
high_priority = (
    support_tickets[support_tickets["priority"] == "High"]
    .groupby("customer_id")
    .size()
    .reset_index()
)
high_priority.columns = ["customer_id", "high_priority_tickets"]

# Merge with support_features
support_features = support_features.merge(high_priority, on="customer_id", how="left")
support_features["high_priority_tickets"] = (
    support_features["high_priority_tickets"].fillna(0).astype(int)
)

print(
    f"Customers with high priority tickets: {(support_features['high_priority_tickets'] > 0).sum()}"
)

Customers with high priority tickets: 758


### Calculate average satisfaction score (excluding 0s which mean no feedback)

In [168]:
# Average satisfaction (excluding 0)
satisfaction = (
    support_tickets[support_tickets["satisfaction_score"] > 0]
    .groupby("customer_id")["satisfaction_score"]
    .mean()
    .reset_index()
)
satisfaction.columns = ["customer_id", "avg_satisfaction"]

# Merge with support_features
support_features = support_features.merge(satisfaction, on="customer_id", how="left")
# Fill NaN with 0 (meaning no satisfaction feedback)
support_features["avg_satisfaction"] = support_features["avg_satisfaction"].fillna(0)

print(f"Average satisfaction statistics:")
print(support_features["avg_satisfaction"].describe())

Average satisfaction statistics:
count   2000.00
mean       3.04
std        1.76
min        0.00
25%        2.00
50%        3.67
75%        4.33
max        5.00
Name: avg_satisfaction, dtype: float64


### Calculate days since last ticket

In [169]:
# Days since last ticket
support_features["days_since_last_ticket"] = (
    observation_date - support_features["last_ticket_date"]
).dt.days

# Drop last_ticket_date
support_features = support_features.drop("last_ticket_date", axis=1)

print(f"Final support features: {support_features.shape}")
print(f"\nColumns: {list(support_features.columns)}")
support_features.head()

Final support features: (2000, 8)

Columns: ['customer_id', 'total_tickets', 'avg_resolution_hours', 'unresolved_tickets', 'technical_tickets_pct', 'high_priority_tickets', 'avg_satisfaction', 'days_since_last_ticket']


,customer_id,total_tickets,avg_resolution_hours,unresolved_tickets,technical_tickets_pct,high_priority_tickets,avg_satisfaction,days_since_last_ticket
0,3,4,20.02,1,25.00,2,0.00,70
1,4,1,12.11,0,100.00,0,4.00,286
2,8,5,17.25,0,40.00,1,4.00,74
3,10,1,40.44,0,0.00,0,0.00,133
4,11,4,12.88,0,25.00,0,4.50,6


---

## Part 4: Join All Features Together

Now we'll combine:
1. Customer features (base)
2. Usage features (left join - some customers may have no usage)
3. Support features (left join - some customers may have no tickets)
4. Churn labels (inner join - all customers should have churn status)

In [170]:
print("Joining all features...")
print("=" * 60)
print(f"Starting with customers: {features.shape}")

# Join usage features (left join)
features = features.merge(usage_features, on="customer_id", how="left")
print(f"After joining usage features: {features.shape}")

# Join support features (left join)
features = features.merge(support_features, on="customer_id", how="left")
print(f"After joining support features: {features.shape}")

# Join churn labels (left join to keep all customers)
features = features.merge(
    churn[["customer_id", "churned"]], on="customer_id", how="left"
)
print(f"After joining churn labels: {features.shape}")

print(f"\nFinal feature count: {len(features.columns)} columns")

Joining all features...
Starting with customers: (5000, 11)
After joining usage features: (5000, 20)
After joining support features: (5000, 27)
After joining churn labels: (5000, 28)

Final feature count: 28 columns


### Handle missing values from left joins

Customers without usage logs or support tickets will have NaN values.
We'll fill these with appropriate defaults (0 for most metrics)

In [171]:
# Check missing values
print("Missing values after joins:")
print(features.isna().sum())

Missing values after joins:
customer_id                    0
signup_date                    0
country                        0
age                            0
gender                         0
subscription_tier              0
monthly_fee                    0
account_age_days               0
is_premium                     0
age_group                      0
country_region                 0
total_sessions                 0
avg_sessions_per_day           0
total_usage_minutes            0
avg_session_duration           0
avg_features_per_session       0
total_errors                   0
days_active                    0
days_since_last_activity       0
usage_consistency_std          0
total_tickets               3000
avg_resolution_hours        3101
unresolved_tickets          3000
technical_tickets_pct       3000
high_priority_tickets       3000
avg_satisfaction            3000
days_since_last_ticket      3000
churned                        0
dtype: int64


In [172]:
# Fill missing usage features with 0 (no usage)
usage_cols = [
    "total_sessions",
    "avg_sessions_per_day",
    "total_usage_minutes",
    "avg_session_duration",
    "avg_features_per_session",
    "total_errors",
    "days_active",
    "days_since_last_activity",
    "usage_consistency_std",
    "usage_trend",
]

for col in usage_cols:
    if col in features.columns:
        features[col] = features[col].fillna(0)

# Fill missing support features with 0 (no tickets)
support_cols = [
    "total_tickets",
    "avg_resolution_hours",
    "unresolved_tickets",
    "technical_tickets_pct",
    "high_priority_tickets",
    "avg_satisfaction",
    "days_since_last_ticket",
]

for col in support_cols:
    if col in features.columns:
        features[col] = features[col].fillna(0)

print("Missing values after filling:")
print(features.isna().sum())

Missing values after filling:
customer_id                 0
signup_date                 0
country                     0
age                         0
gender                      0
subscription_tier           0
monthly_fee                 0
account_age_days            0
is_premium                  0
age_group                   0
country_region              0
total_sessions              0
avg_sessions_per_day        0
total_usage_minutes         0
avg_session_duration        0
avg_features_per_session    0
total_errors                0
days_active                 0
days_since_last_activity    0
usage_consistency_std       0
total_tickets               0
avg_resolution_hours        0
unresolved_tickets          0
technical_tickets_pct       0
high_priority_tickets       0
avg_satisfaction            0
days_since_last_ticket      0
churned                     0
dtype: int64


### Final feature summary

In [173]:
print("Final Features Dataset")
print("=" * 60)
print(f"Shape: {features.shape}")
print(f"\nColumns:")
for i, col in enumerate(features.columns, 1):
    print(f"{i:2d}. {col}")

print(f"\nChurn distribution:")
print(features["churned"].value_counts())
print(f"Churn rate: {features['churned'].mean():.2%}")

Final Features Dataset
Shape: (5000, 28)

Columns:
 1. customer_id
 2. signup_date
 3. country
 4. age
 5. gender
 6. subscription_tier
 7. monthly_fee
 8. account_age_days
 9. is_premium
10. age_group
11. country_region
12. total_sessions
13. avg_sessions_per_day
14. total_usage_minutes
15. avg_session_duration
16. avg_features_per_session
17. total_errors
18. days_active
19. days_since_last_activity
20. usage_consistency_std
21. total_tickets
22. avg_resolution_hours
23. unresolved_tickets
24. technical_tickets_pct
25. high_priority_tickets
26. avg_satisfaction
27. days_since_last_ticket
28. churned

Churn distribution:
churned
0    3750
1    1250
Name: count, dtype: int64
Churn rate: 25.00%


In [174]:
# Show sample of final dataset
features.head(10)

,customer_id,signup_date,country,age,gender,subscription_tier,monthly_fee,account_age_days,is_premium,age_group,country_region,total_sessions,avg_sessions_per_day,total_usage_minutes,avg_session_duration,avg_features_per_session,total_errors,days_active,days_since_last_activity,usage_consistency_std,total_tickets,avg_resolution_hours,unresolved_tickets,technical_tickets_pct,high_priority_tickets,avg_satisfaction,days_since_last_ticket,churned
0,1,2023-10-17,Germany,62.00,F,Basic,13.00,441,0,51+,Europe,349,5.63,1745.06,28.15,8.13,26,62,0,2.65,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
1,2,2022-04-25,UK,43.50,M,Premium,30.48,981,1,36-50,Europe,403,5.30,2515.99,33.11,7.57,39,76,0,2.88,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
2,3,2022-01-26,Brazil,53.00,Unknown,Basic,11.59,1070,0,51+,South America,421,5.40,2312.93,29.65,7.53,42,78,0,2.49,4.00,20.02,1.00,25.00,2.00,0.00,70.00,0
3,4,2024-01-30,Canada,26.00,M,Basic,11.97,336,0,26-35,North America,147,5.25,662.64,23.67,6.86,12,28,4,2.24,1.00,12.11,0.00,100.00,0.00,4.00,286.00,0
4,5,2022-10-09,Spain,44.00,F,Basic,11.23,814,0,36-50,Europe,263,4.61,1803.58,31.64,7.09,37,57,0,2.60,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
5,6,2022-09-08,Brazil,51.00,F,Basic,10.18,845,0,51+,South America,208,4.95,1294.38,30.82,6.74,30,42,1,2.71,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
6,7,2022-08-17,UK,68.00,M,Enterprise,131.87,867,1,51+,Europe,295,5.00,1686.78,28.59,7.95,23,59,0,2.61,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1
7,8,2022-05-23,Germany,18.00,M,Basic,13.95,953,0,18-25,Europe,239,4.88,1741.80,35.55,8.49,22,49,0,2.71,5.00,17.25,0.00,40.00,1.00,4.00,74.00,0
8,9,2024-01-25,Italy,43.50,F,Basic,11.79,341,0,36-50,Europe,113,5.95,641.78,33.78,6.26,6,19,1,2.30,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0
9,10,2022-04-15,USA,42.00,M,Enterprise,132.33,991,1,36-50,North America,105,5.53,395.00,20.79,5.95,12,19,2,2.65,1.00,40.44,0.00,0.00,0.00,0.00,133.00,0


In [175]:
# Show summary statistics
features.describe()

,customer_id,signup_date,age,monthly_fee,account_age_days,is_premium,total_sessions,avg_sessions_per_day,total_usage_minutes,avg_session_duration,avg_features_per_session,total_errors,days_active,days_since_last_activity,usage_consistency_std,total_tickets,avg_resolution_hours,unresolved_tickets,technical_tickets_pct,high_priority_tickets,avg_satisfaction,days_since_last_ticket,churned
count,5000.00,5000,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00,5000.00
mean,2500.50,2023-04-05 01:03:56.160000,43.94,33.48,635.96,0.40,213.01,5.00,1267.20,29.75,7.49,21.30,42.59,2.06,2.57,1.19,6.10,0.24,20.08,0.19,1.22,42.13,0.25
min,1.00,2022-01-01 00:00:00,18.00,9.99,184.00,0.00,15.00,2.50,54.70,9.12,2.00,0.00,5.00,0.00,0.82,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,1250.75,2022-08-29 00:00:00,32.00,12.11,410.75,0.00,124.00,4.71,749.70,27.34,7.05,12.00,25.00,0.00,2.45,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
50%,2500.50,2023-04-06 00:00:00,43.50,14.19,635.00,0.00,209.00,5.00,1239.70,29.66,7.48,20.00,42.00,1.00,2.58,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
75%,3750.25,2023-11-16 06:00:00,57.00,35.24,855.00,1.00,285.00,5.29,1709.34,32.03,7.94,29.00,57.00,2.00,2.70,2.00,11.86,0.00,40.00,0.00,3.00,56.00,0.25
max,5000.00,2024-06-30 00:00:00,70.00,199.97,1095.00,1.00,513.00,8.00,3385.77,64.58,12.17,59.00,89.00,47.00,3.79,5.00,82.01,4.00,100.00,4.00,5.00,365.00,1.00
std,1443.52,NaN,14.90,42.24,260.07,0.49,110.65,0.49,661.99,4.00,0.78,11.85,21.83,4.01,0.24,1.70,9.35,0.56,32.69,0.48,1.86,75.58,0.43


### Save intermediate features dataset

In [176]:
# Save to intermediate folder
features.to_csv("../intermediate/features.csv", index=False)
print("✅ Saved features.csv to intermediate folder")
print(f"   Shape: {features.shape}")
print(f"   Columns: {len(features.columns)}")

✅ Saved features.csv to intermediate folder
   Shape: (5000, 28)
   Columns: 28


---

## Summary

### What We Accomplished in the Intermediate Layer:

✅ **Customer Features**:
- account_age_days - How long customer has been with us
- is_premium - Binary flag for premium tiers
- age_group - Categorical age bins
- country_region - Grouped countries into regions

✅ **Usage Features** (10 features):
- total_sessions, avg_sessions_per_day
- total_usage_minutes, avg_session_duration
- avg_features_per_session
- total_errors
- days_active, days_since_last_activity
- usage_consistency_std (lower = more consistent)
- usage_trend (positive = increasing usage)

✅ **Support Features** (7 features):
- total_tickets, unresolved_tickets
- avg_resolution_hours
- technical_tickets_pct, high_priority_tickets
- avg_satisfaction
- days_since_last_ticket

✅ **Target Variable**:
- churned (0 or 1)

### Key Techniques Used:

1. **Aggregation** - `groupby()` to summarize time-series data
2. **Feature Engineering** - Creating meaningful metrics from raw data
3. **Left Joins** - Preserving all customers even if they have no usage/tickets
4. **Handling Missing Values** - Filling with 0 for customers with no activity
5. **Temporal Features** - Days since last activity, usage trends

### Next Step: Final Layer

Now we'll prepare the final ML-ready dataset:
- Encode categorical variables
- Select relevant features
- Remove any data leakage
- Prepare for train/test split